# DART 재무제표 정합성 자동 검증

OpenDART API로 상장사 연결재무제표를 수집하고, 감사에서 기본으로 확인하는 **재무제표 간 정합성**을 Python으로 자동 검증합니다.

| 검증 항목 | 내용 |
|---|---|
| ① 재무상태표 항등식 | 자산총계 = 부채총계 + 자본총계 |
| ② 순이익 일치 | 손익계산서 당기순이익 = 현금흐름표 당기순이익 |
| ③ 현금 일치 | 현금흐름표 기말 현금 = 재무상태표 현금및현금성자산 |
| ④ 현금 증감 | 기초 현금 + 현금 증감 = 기말 현금 |
| ⑤ 이상 변동 | 재무상태표 계정 중 전기 대비 ±30% 이상 변동 |

계정은 회사마다 이름(`account_nm`)이 조금씩 다르기 때문에, 이름 대신 **XBRL 표준 계정 ID(`account_id`)**로 찾습니다.

## 0. 설정
같은 폴더에 `dart_key.txt` 파일을 만들고 인증키 한 줄만 넣어 두세요. **이 파일은 GitHub에 올리면 안 됩니다** (`.gitignore`에 추가).

In [1]:
import io, zipfile, time
import xml.etree.ElementTree as ET
import requests
import pandas as pd

with open("dart_key.txt", encoding="utf-8") as f:
    API_KEY = f.read().strip()

# 분석 대상: 종목코드 기준 (같은 업종끼리 비교)
TARGETS = {"005930": "삼성전자", "000660": "SK하이닉스", "066570": "LG전자"}
YEARS = [2024, 2025]          # 사업연도
REPRT_CODE = "11011"          # 11011 = 사업보고서(연간)
FS_DIV = "CFS"                # CFS = 연결재무제표, OFS = 별도재무제표
CHANGE_THRESHOLD = 0.30       # 이상 변동 기준 (±30%)

## 1. 기업 고유번호(corp_code) 조회
DART는 종목코드가 아니라 8자리 고유번호로 기업을 구분합니다. 전체 목록(zip 안의 XML)을 받아 종목코드로 찾습니다.

In [2]:
def load_corp_codes(api_key):
    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    res = requests.get(url, params={"crtfc_key": api_key}, timeout=60)
    res.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(res.content)) as z:
        root = ET.fromstring(z.read(z.namelist()[0]))
    rows = [{
        "corp_code": el.findtext("corp_code"),
        "corp_name": el.findtext("corp_name"),
        "stock_code": (el.findtext("stock_code") or "").strip(),
    } for el in root.iter("list")]
    return pd.DataFrame(rows)

corp_df = load_corp_codes(API_KEY)
targets_df = corp_df[corp_df["stock_code"].isin(TARGETS)].reset_index(drop=True)
targets_df

,corp_code,corp_name,stock_code
0,00164779,SK하이닉스,000660
1,00401731,LG전자,066570
2,00126380,삼성전자,005930


## 2. 재무제표 수집
`fnlttSinglAcntAll` API로 회사·연도별 전체 재무제표를 받습니다. `sj_div`는 재무제표 종류입니다 (BS 재무상태표, IS 손익계산서, CIS 포괄손익계산서, CF 현금흐름표, SCE 자본변동표).

In [3]:
def fetch_fs(api_key, corp_code, year):
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {"crtfc_key": api_key, "corp_code": corp_code,
              "bsns_year": str(year), "reprt_code": REPRT_CODE, "fs_div": FS_DIV}
    data = requests.get(url, params=params, timeout=60).json()
    if data.get("status") != "000":          # 000 = 정상, 013 = 데이터 없음
        print(f"[{corp_code} {year}] {data.get('status')} {data.get('message')}")
        return pd.DataFrame()
    return pd.DataFrame(data["list"])

frames = []
for _, row in targets_df.iterrows():
    for year in YEARS:
        df = fetch_fs(API_KEY, row["corp_code"], year)
        if not df.empty:
            df["corp_name"] = row["corp_name"]
            frames.append(df)
        time.sleep(0.3)   # API 호출 간격

raw = pd.concat(frames, ignore_index=True)
raw.to_csv("dart_raw.csv", index=False, encoding="utf-8-sig")
print(raw.shape)
raw[["corp_name", "bsns_year", "sj_div", "account_id", "account_nm", "thstrm_amount", "frmtrm_amount"]].head(10)

(1382, 19)


,corp_name,bsns_year,sj_div,account_id,account_nm,thstrm_amount,frmtrm_amount
0,SK하이닉스,2024,BS,ifrs-full_Assets,자산총계,119855209000000,100330165000000
1,SK하이닉스,2024,BS,ifrs-full_CurrentAssets,유동자산,42278887000000,30468100000000
2,SK하이닉스,2024,BS,ifrs-full_CashAndCashEquivalents,현금및현금성자산,11205117000000,7587329000000
3,SK하이닉스,2024,BS,ifrs-full_CurrentInvestments,단기투자자산,569236000000,860972000000
4,SK하이닉스,2024,BS,ifrs-full_CurrentTaxAssets,당기법인세자산,57467000000,69916000000
5,SK하이닉스,2024,BS,ifrs-full_CurrentTradeReceivables,매출채권,13019006000000,6600273000000
6,SK하이닉스,2024,BS,ifrs-full_Inventories,재고자산,13313937000000,13480659000000
7,SK하이닉스,2024,BS,ifrs-full_NoncurrentAssetsOrDisposalGroupsClas...,매각예정자산,0,29441000000
8,SK하이닉스,2024,BS,ifrs-full_OtherCurrentAssets,기타유동자산,1393744000000,925502000000
9,SK하이닉스,2024,BS,ifrs-full_OtherCurrentFinancialAssets,기타금융자산,45309000000,83750000000


## 3. 정제
- 금액은 문자열로 들어오므로 숫자로 변환합니다 (빈 값 → 결측).
- 같은 표준 계정이라도 회사마다 계정명이 다른지, 표준 계정 ID를 쓰지 않는 회사 고유 계정이 얼마나 되는지 확인합니다.

In [4]:
def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "").str.strip()
                         .replace({"": None, "nan": None, "-": None}), errors="coerce")

fs = raw.copy()
fs["bsns_year"] = fs["bsns_year"].astype(int)
for col in ["thstrm_amount", "frmtrm_amount"]:
    fs[col] = to_num(fs[col])
fs["account_id_l"] = fs["account_id"].str.lower()

# (1) 같은 표준 계정 ID, 회사별 계정명 비교
key_ids = ["ifrs-full_assets", "ifrs-full_profitloss", "ifrs-full_cashandcashequivalents"]
name_check = (fs[fs["account_id_l"].isin(key_ids)]
              .groupby(["account_id", "corp_name"])["account_nm"].unique().unstack())
display(name_check)

# (2) 표준 계정 ID 미사용 비율 (회사 고유 계정)
nonstd = (fs.assign(nonstd=~fs["account_id_l"].str.contains("ifrs|dart", na=False))
            .groupby(["corp_name", "bsns_year"])["nonstd"].mean().round(3))
nonstd.rename("표준계정 미사용 비율")

corp_name,LG전자,SK하이닉스,삼성전자
account_id,,,
ifrs-full_Assets,[자산총계],[자산총계],[자산총계]
ifrs-full_CashAndCashEquivalents,[현금및현금성자산],[현금및현금성자산],[현금및현금성자산]
ifrs-full_ProfitLoss,[당기순이익(손실)],[당기순이익(손실)],[당기순이익]


corp_name  bsns_year
LG전자       2024         0.054
           2025         0.074
SK하이닉스     2024         0.085
           2025         0.017
삼성전자       2024         0.033
           2025         0.022
Name: 표준계정 미사용 비율, dtype: float64

## 4. 정합성 검증

In [5]:
def pick(df, sj, ids=None, name_has=None, col="thstrm_amount"):
    """재무제표 종류(sj) 안에서 표준 계정 ID 또는 계정명으로 금액 하나를 찾는다."""
    d = df[df["sj_div"].isin(sj)]
    if ids:
        hit = d[d["account_id_l"].isin([i.lower() for i in ids])]
        if not hit.empty:
            return hit[col].iloc[0]
    if name_has:
        nm = d["account_nm"].str.replace(" ", "")
        hit = d[nm.apply(lambda x: all(k in x for k in name_has))]
        if not hit.empty:
            return hit[col].iloc[0]
    return None

def check(name, left, right):
    if left is None or right is None or pd.isna(left) or pd.isna(right):
        return {"검증": name, "좌변": left, "우변": right, "차이": None, "결과": "확인불가"}
    diff = left - right
    return {"검증": name, "좌변": left, "우변": right, "차이": diff,
            "결과": "통과" if abs(diff) < 1 else "불일치"}

results = []
for (corp, year), d in fs.groupby(["corp_name", "bsns_year"]):
    assets = pick(d, ["BS"], ["ifrs-full_Assets"], ["자산총계"])
    liab   = pick(d, ["BS"], ["ifrs-full_Liabilities"], ["부채총계"])
    equity = pick(d, ["BS"], ["ifrs-full_Equity"], ["자본총계"])
    ni_is  = pick(d, ["IS", "CIS"], ["ifrs-full_ProfitLoss"], ["당기순이익"])
    ni_cf  = pick(d, ["CF"], ["ifrs-full_ProfitLoss"], ["당기순이익"])
    cash_bs  = pick(d, ["BS"], ["ifrs-full_CashAndCashEquivalents"], ["현금및현금성자산"])
    cash_end = pick(d, ["CF"], None, ["기말", "현금"])
    cash_beg = pick(d, ["CF"], None, ["기초", "현금"])
    cash_chg = pick(d, ["CF"], ["ifrs-full_IncreaseDecreaseInCashAndCashEquivalents"],
                    ["현금", "증가"])
    checks = [
        check("① 자산 = 부채 + 자본", assets,
              None if liab is None or equity is None else liab + equity),
        check("② 순이익(손익) = 순이익(현금흐름)", ni_is, ni_cf),
        check("③ 기말현금(현금흐름) = 현금(재무상태)", cash_end, cash_bs),
        check("④ 기초현금 + 증감 = 기말현금",
              None if cash_beg is None or cash_chg is None else cash_beg + cash_chg, cash_end),
    ]
    for c in checks:
        results.append({"회사": corp, "연도": year, **c})

result_df = pd.DataFrame(results)
result_df.to_csv("dart_check_result.csv", index=False, encoding="utf-8-sig")
result_df

,회사,연도,검증,좌변,우변,차이,결과
0,LG전자,2024,① 자산 = 부채 + 자본,6.562959e+13,6.562959e+13,0.0,통과
1,LG전자,2024,② 순이익(손익) = 순이익(현금흐름),5.913650e+11,NaN,NaN,확인불가
2,LG전자,2024,③ 기말현금(현금흐름) = 현금(재무상태),7.573036e+12,7.573036e+12,0.0,통과
3,LG전자,2024,④ 기초현금 + 증감 = 기말현금,7.573036e+12,7.573036e+12,0.0,통과
4,LG전자,2025,① 자산 = 부채 + 자본,6.862017e+13,6.862017e+13,0.0,통과
5,LG전자,2025,② 순이익(손익) = 순이익(현금흐름),1.220412e+12,NaN,NaN,확인불가
6,LG전자,2025,③ 기말현금(현금흐름) = 현금(재무상태),8.769841e+12,8.769841e+12,0.0,통과
7,LG전자,2025,④ 기초현금 + 증감 = 기말현금,8.769841e+12,8.769841e+12,0.0,통과
8,SK하이닉스,2024,① 자산 = 부채 + 자본,1.198552e+14,1.198552e+14,0.0,통과
9,SK하이닉스,2024,② 순이익(손익) = 순이익(현금흐름),1.979690e+13,NaN,NaN,확인불가


In [6]:
# 결과 요약: 회사·연도별 통과 개수
result_df.pivot_table(index=["회사", "연도"], columns="결과", values="검증",
                      aggfunc="count", fill_value=0)

결과           통과  확인불가
회사     연도            
LG전자   2024   3     1
       2025   3     1
SK하이닉스 2024   3     1
       2025   3     1
삼성전자   2024   4     0
       2025   4     0

## 5. 이상 변동 탐지
재무상태표에서 전기 대비 변동률이 ±30% 이상인 계정을 찾습니다. 금액이 너무 작은 계정은 변동률이 과장되므로 자산총계의 1% 이상인 계정만 봅니다.

In [7]:
bs = fs[fs["sj_div"] == "BS"].copy()
bs["변동률"] = (bs["thstrm_amount"] - bs["frmtrm_amount"]) / bs["frmtrm_amount"].abs()

total_assets = (bs[bs["account_id_l"] == "ifrs-full_assets"]
                .set_index(["corp_name", "bsns_year"])["thstrm_amount"])
bs["자산대비"] = bs.apply(lambda r: r["thstrm_amount"] / total_assets.get((r["corp_name"], r["bsns_year"]), float("nan")), axis=1)

anomaly = bs[(bs["변동률"].abs() >= CHANGE_THRESHOLD) & (bs["자산대비"].abs() >= 0.01)]
anomaly = anomaly[["corp_name", "bsns_year", "account_nm", "frmtrm_amount", "thstrm_amount", "변동률"]]
anomaly = (anomaly.assign(_abs=anomaly["변동률"].abs())
           .sort_values(["corp_name", "bsns_year", "_abs"], ascending=[True, True, False])
           .drop(columns="_abs"))
anomaly["변동률"] = (anomaly["변동률"] * 100).round(1).astype(str) + "%"
anomaly.to_csv("dart_anomaly.csv", index=False, encoding="utf-8-sig")
anomaly

,corp_name,bsns_year,account_nm,frmtrm_amount,thstrm_amount,변동률
494,LG전자,2024,기타포괄손익누계액,-4.208180e+11,8.129130e+11,293.2%
468,LG전자,2024,기타수취채권,5.957930e+11,1.583477e+12,165.8%
487,LG전자,2024,비유동매출채권,6.394390e+11,1.645716e+12,157.4%
507,LG전자,2024,충당부채,1.049625e+12,1.515367e+12,44.4%
480,LG전자,2024,기타수취채권,1.779637e+12,1.022169e+12,-42.6%
696,LG전자,2025,기타자본구성요소,-1.628080e+11,1.240375e+12,861.9%
719,LG전자,2025,계약부채,1.001770e+11,9.041440e+11,802.5%
689,LG전자,2025,매출채권,1.645716e+12,2.783685e+12,69.1%
688,LG전자,2025,순확정급여자산,5.070870e+11,7.997710e+11,57.7%
697,LG전자,2025,기타포괄손익누계액,8.129130e+11,1.234223e+12,51.8%


In [8]:
fs[(fs["corp_name"] == "SK하이닉스") & (fs["sj_div"] == "CF") & (fs["bsns_year"] == 2025)][["account_nm"]].head(8)

,account_nm
317,기초 현금및현금성자산
318,기말 현금및현금성자산
319,재무활동 현금흐름
320,자기주식의 처분
321,주식선택권의 행사
322,종속기업 유상증자 및 소유지분변동
323,배당금의 지급
324,리스부채의 상환


In [ ]:
# ② 확인불가 원인 확인: 회사별 현금흐름표(2025)에 어떤 행이 있는지 비교
cf = fs[(fs["sj_div"] == "CF") & (fs["bsns_year"] == 2025)]
has = lambda aid: cf.groupby("corp_name")["account_id_l"].apply(lambda s: (s == aid).any())
pd.DataFrame({
    "당기순이익 행": has("ifrs-full_profitloss"),
    "영업에서 창출된 현금 행": has("ifrs-full_cashflowsfromusedinoperations"),
})

,당기순이익 행,영업에서 창출된 현금 행
corp_name,,
LG전자,False,True
SK하이닉스,False,True
삼성전자,True,True


## 6. 해석

### 검증 결과
- 3개 기업 × 2개년 × 4개 규칙 = **24건 중 통과 20건, 불일치 0건, 확인불가 4건**.
- 공시된 재무제표는 이미 외부감사를 받은 자료이므로 불일치가 없는 것이 정상입니다. 이 노트북의 의미는 오류를 찾은 것이 아니라, **재무제표 간 정합성을 자동으로 점검하는 틀**을 만든 데 있습니다. 같은 틀을 검증 전 내부 데이터에 적용하면 입력·집계 오류를 잡을 수 있습니다.

### 확인불가 4건의 원인
- 모두 ② 순이익 일치 검증이며, **LG전자와 SK하이닉스**에서만 발생했습니다.
- 위 비교표처럼 두 회사의 현금흐름표 본문은 당기순이익 행 없이 **"영업으로부터 창출된 현금흐름"**에서 시작합니다(당기순이익 조정 내역은 주석에 표시). 삼성전자는 본문에 당기순이익과 조정 항목을 직접 표시해 검증이 가능했습니다.
- 즉, 회사마다 현금흐름표 작성 방식이 달라 **재무제표 본문만으로는 검증할 수 없는 항목**이 있습니다. 이 항목을 검증하려면 주석 데이터가 필요합니다.

### 정제하며 발견한 점
- **같은 이름의 계정 중복**: LG전자 2024년 재무상태표에 `기타수취채권`이 두 행(유동·비유동) 있습니다. 계정명으로 찾으면 엉뚱한 행을 집을 수 있습니다.
- **연도별로 바뀐 계정명**: LG전자의 비유동 매출채권(약 1.65조 원)은 2024년 보고서에서 `비유동매출채권`, 2025년 보고서에서는 `매출채권`으로 표기됐습니다. 그래서 2025년에는 유동·비유동 매출채권이 같은 이름입니다. 계정명은 바뀌었지만 표준 계정 ID(`ifrs-full_NoncurrentTradeReceivables`)는 같아서, ID로 찾으면 같은 계정으로 이어집니다.
- 이런 이유로 계정명 대신 **XBRL 표준 계정 ID(`account_id`)**로 계정을 찾았습니다. 다만 LG전자 기타수취채권은 ID 자체가 2024년 `dart_CurrentNontradeReceivables`에서 2025년 `ifrs-full_OtherCurrentReceivables`로 바뀌어, 연도 간 비교 시에는 ID 매핑도 확인이 필요합니다. 표준 계정 ID를 쓰지 않는 회사 고유 계정 비율은 삼성전자 2.7%, SK하이닉스 5.2%, LG전자 6.5%였습니다.

### 이상 변동 (전기 대비 ±30% 이상, 자산총계의 1% 이상 계정)
- **SK하이닉스 2024**: 당기법인세부채 약 439억 원 → 약 3.08조 원(약 70배), 매출채권 +97%, 차입금 -47%.
- **SK하이닉스 2025**: 단기투자자산 +838%, 단기금융상품 +516%, 장기투자자산 +260%. 늘어난 현금이 금융상품으로 운용되는 흐름이 보입니다.
- 변동의 실제 원인은 사업보고서 주석으로 확인해야 하며, 이 단계는 **감사에서 우선 살펴볼 계정을 골라내는 용도**입니다.

### 한계와 개선점
- **부호가 바뀌는 계정은 변동률이 왜곡됩니다.** 예: LG전자 2024 기타포괄손익누계액 -4,208억 원 → +8,129억 원(293%). 부호 전환 계정은 따로 분리해 표시해야 합니다.
- 현금흐름표 순이익 검증은 주석 데이터까지 수집하면 모든 회사에 적용할 수 있습니다.